# Using Iterators in PythonLand — OOP Edition

**Theme:** financial market-data engineering and financial-news intelligence.

All 13 requested subtopics are covered in order. The examples use classes, methods, custom iterators, generators, and streaming pipelines rather than isolated toy examples.

## Learning Map

`Iterable → Custom Iterator → for loop → iterator functions → enumerate() → zip() → unzip → file streaming → generator methods → text extraction → streaming analytics`

## 1. Introduction to iterators

**Concept:** An iterator produces values sequentially while preserving its current position.

In [ ]:
class PriceSeries:
    def __init__(self, ticker, prices):
        self.ticker = ticker
        self.prices = prices

    def iterator(self):
        return iter(self.prices)

    def average_price(self):
        return sum(self.prices) / len(self.prices)


bhp = PriceSeries(
    "BHP",
    [41.70, 42.15, 41.92, 43.08, 42.76, 44.21]
)

prices = bhp.iterator()

print(bhp.ticker)
print("First:", next(prices))
print("Second:", next(prices))
print("Average:", round(bhp.average_price(), 2))

## 2. Iterators vs. Iterables

**Concept:** An iterable can create an iterator, while an iterator also stores traversal state and implements `__next__()`.


In [2]:
class Portfolio:
    def __init__(self, positions):
        self.positions = positions

    def __iter__(self):
        return iter(self.positions.items())


portfolio = Portfolio({
    "BHP": 0.18,
    "CBA": 0.27,
    "CSL": 0.20,
    "RIO": 0.20,
    "WES": 0.15
})

print("Iterable:", hasattr(portfolio, "__iter__"))
print("Iterator:", hasattr(portfolio, "__next__"))

portfolio_iterator = iter(portfolio)

print(next(portfolio_iterator))
print(next(portfolio_iterator))
print(list(portfolio_iterator))

Iterable: True
Iterator: False
('BHP', 0.18)
('CBA', 0.27)
[('CSL', 0.2), ('RIO', 0.2), ('WES', 0.15)]


## 3. Iterating over iterables (1)

**Concept:** A `for` loop automatically calls `__iter__()` and consumes the resulting iterator.

In [4]:
class TradeBook:
    def __init__(self, trades):
        self.trades = trades

    def __iter__(self):
        return iter(self.trades)

    def total_notional(self):
        return sum(
            trade["quantity"] * trade["price"]
            for trade in self
        )


trade_book = TradeBook([
    {"ticker": "BHP", "side": "BUY", "quantity": 2400, "price": 41.70},
    {"ticker": "CBA", "side": "SELL", "quantity": 800, "price": 178.25},
    {"ticker": "CSL", "side": "BUY", "quantity": 350, "price": 312.40},
    {"ticker": "RIO", "side": "BUY", "quantity": 1200, "price": 119.80},
])

for trade in trade_book:
    notional = trade["quantity"] * trade["price"]
    print(f'{trade["side"]:4} {trade["ticker"]:4} ${notional:,.2f}')

print(f"Total notional: ${trade_book.total_notional():,.2f}")

BUY  BHP  $100,080.00
SELL CBA  $142,600.00
BUY  CSL  $109,340.00
BUY  RIO  $143,760.00
Total notional: $495,780.00


## 4. Iterating over iterables (2)

**Concept:** An iterable class can expose business-level traversal over nested data without exposing its internal representation.

In [6]:
class MarketSnapshot:
    def __init__(self, ticker, metrics, venues):
        self.ticker = ticker
        self.metrics = metrics
        self.venues = venues

    def __iter__(self):
        for metric, value in self.metrics.items():
            yield metric, value

    def describe(self):
        for metric, value in self:
            print(f"{self.ticker} | {metric:<12} | {value}")


snapshot = MarketSnapshot(
    "BHP",
    {
        "price": 41.70,
        "volume": 1_842_000,
        "spread_bps": 3.7,
        "volatility": 0.284
    },
    ("ASX", "Chi-X")
)

snapshot.describe()

for venue in snapshot.venues:
    print("Venue:", venue)

BHP | price        | 41.7
BHP | volume       | 1842000
BHP | spread_bps   | 3.7
BHP | volatility   | 0.284
Venue: ASX
Venue: Chi-X
